# Apex Retail Intelligence

## KPI Dashboard

### Objective
Generate business insights from Gold Layer.

## Project Pipeline

```text
                Apex Retail Intelligence Pipeline

 Historical CSV          Incremental CSV
        │                      │
        └──────────┬───────────┘
                   ▼
            Raw Landing Layer
                   │
                   ▼
             Bronze Layer
            (Delta Storage)
                   │
                   ▼
              Silver Layer
      (Cleaning + MERGE + SCD)
                   │
                   ▼
               Gold Layer
      (Dimensions & Fact Tables)
                   │
                   ▼
              KPI Dashboard
```

## Step 1 : Load Gold Tables

In this step, I am loading the Gold Layer tables into PySpark DataFrames.

These tables will be used to calculate business KPIs.

In [0]:
# ============================================================
# Step 1 : Load Gold Tables
# ============================================================

dim_customer_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_customer"
)

dim_product_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_product"
)

dim_date_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/dim_date"
)

fact_sales_df = spark.read.format("delta").load(
    "dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/04_Gold/fact_sales"
)

print("Gold tables loaded successfully.")

Gold tables loaded successfully.


## Step 2 : Preview Gold Tables

In this step, I am displaying the Gold Layer tables to verify that they have been loaded successfully.

In [0]:
# ============================================================
# Step 2 : Preview Gold Tables
# ============================================================

display(dim_customer_df.limit(5))

display(dim_product_df.limit(5))

display(dim_date_df.limit(5))

display(fact_sales_df.limit(5))

customer_sk,customer_id,age,gender,income_bracket,customer_city,customer_state
0,1,56,Other,High,City D,State Y
1,2,69,Female,Medium,null,State X
2,3,46,Female,Low,City B,State X
3,4,32,Female,Low,City A,State Y
4,6,25,Other,Medium,City D,State Z


product_sk,product_id,product_name,product_brand,product_category,unit_price
0,100,Product D,Brand Y,Toys,629.45
1,10000,Product E,Brand Y,Furniture,914.04
2,10001,Product B,Brand X,Groceries,446.16
3,10002,Product C,Brand Y,Toys,523.74
4,10003,Product D,Brand Y,Clothing,467.44


transaction_date,date,year,month,week,day,day_of_week
2021-03-09 04:28:26,2021-03-09,2021,3,10,9,3
2021-03-10 18:01:53,2021-03-10,2021,3,10,10,4
2022-03-18 20:09:55,2022-03-18,2022,3,11,18,6
2023-09-13 21:48:59,2023-09-13,2023,9,37,13,4
2023-12-11 19:43:03,2023-12-11,2023,12,50,11,2


sales_sk,transaction_id,customer_id,product_id,transaction_date,quantity,unit_price,discount_applied,total_sales
0,191823,23,8715,2021-11-08 21:20:37,9,512.97,0.02,8820.87
1,974720,26,414,2020-02-08 02:45:45,2,687.35,0.04,1639.63
2,577038,29,4642,2021-03-27 13:44:19,5,833.11,0.32,1459.27
3,765484,48,8087,2020-07-16 08:00:45,8,719.79,0.13,8729.63
4,795563,92,8534,2020-02-21 21:15:33,8,607.74,0.5,1340.42


## Step 3 : Calculate Total Sales

In this step, I am calculating the total sales amount.

In [0]:
# ============================================================
# Step 3 : Total Sales
# ============================================================

from pyspark.sql.functions import sum

total_sales = fact_sales_df.select(
    sum("total_sales")
).collect()[0][0]

print("Total Sales :", total_sales)

Total Sales : 5525601.659999984


## Step 4 : Total Customers

In this step, I am calculating the total number of customers.

In [0]:
# ============================================================
# Step 4 : Total Customers
# ============================================================

print("Total Customers :", dim_customer_df.count())

Total Customers : 1050


## Step 5 : Total Products

In this step, I am calculating the total number of products.

In [0]:
# ============================================================
# Step 5 : Total Products
# ============================================================

print("Total Products :", dim_product_df.count())

Total Products : 1043


## Step 6 : Calculate Average Sales

In this step, I am calculating the average sales amount per transaction.

In [0]:
# ============================================================
# Step 6 : Average Sales
# ============================================================

from pyspark.sql.functions import avg

average_sales = fact_sales_df.select(
    avg("total_sales")
).collect()[0][0]

print("Average Sales :", average_sales)

Average Sales : 2882.4213145539825


## Step 7 : Top 10 Products

In this step, I am identifying the top 10 products based on total sales.

In [0]:
# ============================================================
# Step 7 : Top 10 Products
# ============================================================

from pyspark.sql.functions import sum

top_products = (

    fact_sales_df

    .groupBy("product_id")

    .agg(
        sum("total_sales").alias("Total Sales")
    )

    .orderBy(
        "Total Sales",
        ascending=False
    )

    .limit(10)

)

display(top_products)

product_id,Total Sales
2168,17065.35
2012,16418.06
3647,15436.939999999999
7931,15414.91
83,13959.57
6789,13776.93
9727,13629.749999999998
4355,13497.929999999998
616,13129.09
8109,13047.02


Databricks visualization. Run in Databricks to view.

## Step 8 : Top 10 Customers

In this step, I am identifying the top 10 customers based on total sales.

In [0]:
# ============================================================
# Step 8 : Top 10 Customers
# ============================================================

top_customers = (

    fact_sales_df

    .groupBy("customer_id")

    .agg(
        sum("total_sales").alias("Total Sales")
    )

    .orderBy(
        "Total Sales",
        ascending=False
    )

    .limit(10)

)

display(top_customers)

customer_id,Total Sales
110,25096.329999999998
415,23652.03
273,21471.93
236,20469.11
374,19704.43
427,18531.07
244,18190.83
160,17988.33
378,17836.739999999998
11,17527.18


Databricks visualization. Run in Databricks to view.

## Step 9 : Monthly Sales

In this step, I am calculating monthly sales for business reporting.

In [0]:
# ============================================================
# Step 9 : Monthly Sales
# ============================================================

from pyspark.sql.functions import month

monthly_sales = (

    fact_sales_df

    .withColumn(
        "Month",
        month("transaction_date")
    )

    .groupBy("Month")

    .agg(
        sum("total_sales").alias("Total Sales")
    )

    .orderBy("Month")

)

display(monthly_sales)

Month,Total Sales
null,145922.28999999995
1,408728.6400000003
2,454301.6
3,460644.4199999999
4,456350.4300000002
5,400179.19999999995
6,434222.0900000001
7,477512.0099999996
8,426677.13999999996
9,449149.1499999998


Databricks visualization. Run in Databricks to view.

## Step 10 : KPI Summary

The KPI analysis has been completed successfully.

The following business metrics were generated:

- Total Sales
- Average Sales
- Total Customers
- Total Products
- Top 10 Products
- Top 10 Customers
- Monthly Sales

The Apex Retail Intelligence project has been completed successfully.

In [0]:
# ============================================================
# Step 10 : KPI Summary
# ============================================================

print("KPI Notebook completed successfully.")

print("Total Sales      :", total_sales)
print("Average Sales    :", average_sales)
print("Total Customers  :", dim_customer_df.count())
print("Total Products   :", dim_product_df.count())

print("\nBusiness KPI generation completed successfully.")

KPI Notebook completed successfully.
Total Sales      : 5525601.659999984
Average Sales    : 2882.4213145539825
Total Customers  : 1050
Total Products   : 1043

Business KPI generation completed successfully.


## Project Completed

✔ Total Sales Calculated

✔ Average Sales Calculated

✔ Top Products Generated

✔ Monthly Sales Generated

✔ Apex Retail Intelligence Pipeline Completed Successfully